# Chapter 30: Moving to GPU

        **Part V - The Training Loop, Mastered** · Companion notebook for *PyTorch From Ground Up, Volume I: Foundations*

        This notebook follows the chapter's progression with fresh, CPU-friendly examples. Type, change, break, and repair the code; the goal is fluency, not passive reading.

        ## Learning objectives

        - Select an available accelerator
- Move model and batches consistently
- Save and load device-portable weights


In [ ]:
import torch

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Default device: cpu")


## Detect once

Create one device variable and use it everywhere. CPU remains the universal fallback.


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("selected:", device)


## Move model and batches together

Operations require participating tensors on the same device. Transfer each batch inside the loop.


In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

loader = DataLoader(TensorDataset(torch.randn(64, 5), torch.randint(0, 2, (64,))), batch_size=16)
model = nn.Linear(5, 2).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
for batch_x, batch_y in loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
    loss = nn.functional.cross_entropy(model(batch_x), batch_y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
print("model device:", next(model.parameters()).device)


## Return results and load portably

Move tensors back to CPU before NumPy conversion. map_location makes a saved state portable across machines.


In [ ]:
import tempfile
from pathlib import Path

with torch.no_grad():
    cpu_probabilities = model(batch_x).softmax(1).cpu()
path = Path(tempfile.mkdtemp(prefix="pytorch-device-")) / "weights.pt"
torch.save(model.state_dict(), path)
cpu_model = nn.Linear(5, 2)
cpu_model.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
print(cpu_probabilities.device, next(cpu_model.parameters()).device)


## Practice

        Work through these without looking back first.

        1. Print whether CUDA or MPS is available on your machine.
2. Trigger and explain a device mismatch if an accelerator is available.
3. Load saved weights explicitly onto CPU.

        <details><summary>Study habit</summary>

        Predict every output shape before running a cell. When something fails, read the final line of the error and print the shape, dtype, and device of every tensor involved.

        </details>


In [ ]:
# Your practice space. Replace `pass` with your solution.
pass


## Recap

You completed Chapter 30's companion lab. Before moving on, explain the central idea aloud and recreate the smallest useful example from memory.

**Next:** return to the book for the full explanation, visual reasoning, watch-outs, and chapter exercises.
